In [7]:
import pandas as pd
from datetime import datetime, timedelta

csv_file_path = "//10.69.168.1/crnldata/forgetting/Carla/Fed/APPPS1/AHAD11.71/7 mois/AHAD11.71_05_01_2026au07_01_2026.CSV"

def transform_dates():
    try:
        # Lecture avec point-virgule comme séparateur
        print("Lecture du fichier...")
        df = pd.read_csv(csv_file_path, sep=';')
        
        print(f"Nombre de colonnes: {len(df.columns)}")
        print(f"Colonnes: {df.columns.tolist()}")
        print(f"\nPremière ligne:")
        print(df.head(1))
        
        # La première colonne contient les dates
        date_column = df.columns[0]
        print(f"\nColonne de date: '{date_column}'")
        print(f"Exemple avant transformation: {df[date_column].iloc[0]}")
        
        # Pour gérer "0/0/2000 0:00:09", on remplace par "1/1/2000 0:00:09"
        # Car 0/0 n'est pas une date valide
        old_reference_str = "0/0/2000 0:00:09"
        # Remplacer 0/0 par 1/1
        old_reference_corrected = old_reference_str.replace("0/0/", "1/1/")
        old_reference = datetime.strptime(old_reference_corrected, "%d/%m/%Y %H:%M:%S")
        new_reference = datetime.strptime("1/5/2026 15:32:45", "%d/%m/%Y %H:%M:%S")
        
        time_offset = new_reference - old_reference
        print(f"Décalage temporel: {time_offset}")
        
        # Fonction de transformation
        def transform_single_date(date_str):
            if pd.isna(date_str):
                return date_str
            
            try:
                date_str_clean = str(date_str).strip()
                
                # Gérer le cas spécial "0/0/YYYY" -> le remplacer par "1/1/YYYY"
                if date_str_clean.startswith("0/0/"):
                    date_str_clean = date_str_clean.replace("0/0/", "1/1/", 1)
                    print(f"Date corrigée: 0/0/... -> {date_str_clean}")
                
                # Essayer différents formats de date
                for fmt in ["%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y"]:
                    try:
                        parsed_date = datetime.strptime(date_str_clean, fmt)
                        new_date = parsed_date + time_offset
                        
                        # Retourner dans le même format que l'input
                        if ":" in date_str_clean:
                            if date_str_clean.count(":") == 2:
                                return new_date.strftime("%d/%m/%Y %H:%M:%S")
                            else:
                                return new_date.strftime("%d/%m/%Y %H:%M")
                        else:
                            return new_date.strftime("%d/%m/%Y")
                    except ValueError:
                        continue
                
                # Si aucun format ne fonctionne
                print(f"Format non reconnu pour '{date_str_clean}'")
                return date_str
                
            except Exception as e:
                print(f"Erreur sur '{date_str_clean}': {e}")
                return date_str
        
        # Applique la transformation
        print("\nTransformation en cours...")
        df[date_column] = df[date_column].apply(transform_single_date)
        
        print(f"Exemple après transformation: {df[date_column].iloc[0]}")
        print(f"\nAperçu des données transformées:")
        print(df.head(5))
        
        # Sauvegarde avec point-virgule comme séparateur
        output_path = csv_file_path.replace(".CSV", "_transformed.CSV")
        print(f"\nSauvegarde du fichier vers: {output_path}")
        df.to_csv(output_path, index=False, sep=';')
        
        print("✓ Fichier transformé et sauvegardé!")
        print(f"✓ Fichier original préservé")
        print(f"✓ Nouveau fichier: {output_path}")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    transform_dates()

Lecture du fichier...
Nombre de colonnes: 16
Colonnes: ['MM:DD:YYYY hh:mm:ss', 'Library_Version', 'Session_type', 'Device_Number', 'Battery_Voltage', 'Motor_Turns', 'FR', 'Event', 'Active_Poke', 'Left_Poke_Count', 'Right_Poke_Count', 'Pellet_Count', 'Block_Pellet_Count', 'Retrieval_Time', 'InterPelletInterval', 'Poke_Time']

Première ligne:
  MM:DD:YYYY hh:mm:ss Library_Version Session_type  Device_Number  \
0    0/0/2000 0:00:09          1.16.3    Light Trk              0   

   Battery_Voltage  Motor_Turns  FR Event Active_Poke  Left_Poke_Count  \
0             4.15          NaN   1  Left        Left                1   

   Right_Poke_Count  Pellet_Count  Block_Pellet_Count Retrieval_Time  \
0                 0             0                   0            NaN   

   InterPelletInterval  Poke_Time  
0                  NaN        0.0  

Colonne de date: 'MM:DD:YYYY hh:mm:ss'
Exemple avant transformation: 0/0/2000 0:00:09
Décalage temporel: 9617 days, 15:32:36

Transformation en cours..